# Demo Pessoa 1 - TSP Medico com Algoritmo Genetico

Este notebook demonstra a camada de dominio do problema: catalogo de pontos, matriz de distancia, decoder da rota com abastecimento automatico e funcao de fitness.

A parte de selecao, crossover, mutacao, elitismo e loop evolutivo fica fora deste notebook, pois sera implementada pela Pessoa 2.

## 1. Preparacao do ambiente

O notebook adiciona a raiz do projeto ao `sys.path` para importar os modulos de `src/` ao executar a partir da pasta `notebooks/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_CONFIG
from src.data_loader import get_hospitals, get_origin, get_supply_stations, load_points
from src.distance import build_distance_matrix
from src.fitness import evaluate
from src.models import Config

## 2. Carregamento do catalogo

O catalogo possui uma origem, hospitais obrigatorios e pontos de abastecimento. O algoritmo genetico manipula apenas os indices dos hospitais.

In [ ]:
points = load_points(PROJECT_ROOT / "data" / "pontos_entrega.csv")
origin = get_origin(points)
hospitals = get_hospitals(points)
supply_stations = get_supply_stations(points)

print(f"Origem: {origin.idx} - {origin.name}")
print(f"Hospitais obrigatorios: {len(hospitals)}")
print(f"Pontos de abastecimento: {len(supply_stations)}")

In [ ]:
for point in points:
    print(
        f"{point.idx:>3} | {point.type:<8} | {point.name:<24} | "
        f"prioridade={str(point.priority):<5} | demanda={point.demand}"
    )

## 3. Matriz de distancia

As distancias sao calculadas uma unica vez por Haversine e ficam indexadas por pares `(origem, destino)`.

In [ ]:
distance_matrix = build_distance_matrix(points)

print(f"Total de pares na matriz: {len(distance_matrix)}")
print(f"Distancia Hospital Central -> Unidade A: {distance_matrix[(0, 1)]:.2f} km")

## 4. Avaliacao de um cromossomo

O cromossomo contem somente hospitais. Origem e abastecimentos sao inseridos automaticamente pelo decoder.

In [ ]:
def print_result(title, result):
    print(title)
    print(f"Cromossomo: {result.chromosome}")
    print(f"Rota decodificada: {result.decoded_route}")
    print(f"Distancia total: {result.total_distance:.2f} km")
    print(f"Penalidade de prioridade: {result.priority_penalty:.2f}")
    print(f"Penalidade de abastecimento: {result.supply_penalty:.2f}")
    print(f"Reabastecimentos: {result.resupply_count}")
    print(f"Fitness: {result.fitness:.2f}")
    print(f"Valido: {result.is_valid}")
    if result.errors:
        print(f"Erros: {result.errors}")

In [ ]:
chromosome = [3, 1, 5, 2, 4]
result = evaluate(chromosome, points, distance_matrix, DEFAULT_CONFIG)

print_result("Exemplo principal", result)

## 5. Comparacao de prioridade

A prioridade ALTA tem peso maior. Quando aparece tarde na rota, a penalidade aumenta.

In [ ]:
alta_mais_cedo = evaluate([1, 3, 2], points, distance_matrix, DEFAULT_CONFIG)
alta_mais_tarde = evaluate([3, 2, 1], points, distance_matrix, DEFAULT_CONFIG)

print_result("ALTA mais cedo", alta_mais_cedo)
print()
print_result("ALTA mais tarde", alta_mais_tarde)

## 6. Cenario com abastecimento forcado

Reduzindo a capacidade do veiculo, o decoder precisa inserir automaticamente uma unidade de abastecimento antes da proxima entrega.

In [ ]:
low_capacity_config = Config(vehicle_capacity=50, lambda_priority=5.0, lambda_supply=10.0)
resupply_result = evaluate([7, 10], points, distance_matrix, low_capacity_config)

print_result("Cenario com abastecimento", resupply_result)

## 7. Exemplo de cromossomo invalido

A funcao `evaluate()` sempre retorna um `EvaluationResult`. Quando o cromossomo e invalido, `is_valid=False` e `errors` explica o problema.

In [ ]:
invalid_chromosome = [0, 1, 1, 100]
invalid_result = evaluate(invalid_chromosome, points, distance_matrix, DEFAULT_CONFIG)

print_result("Cromossomo invalido", invalid_result)

## Interface final para a Pessoa 2

A Pessoa 2 precisa apenas construir ou receber um cromossomo `list[int]` e chamar:

```python
result = evaluate(chromosome, points, distance_matrix, config)
fitness = result.fitness
```

O algoritmo genetico continua livre para implementar selecao, crossover, mutacao, elitismo e loop evolutivo usando esse valor de fitness.